# Fine-tuning a masked language model (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

You will need to setup git, adapt your email and name in the following cell.

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
from transformers import AutoModelForMaskedLM

model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> DistilBERT number of parameters: {round(distilbert_num_parameters)}M'")
print(f"'>>> BERT number of parameters: 110M'")

'>>> DistilBERT number of parameters: 67M'
'>>> BERT number of parameters: 110M'


In [4]:
text = "This is a great [MASK]."

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [6]:
import torch

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great deal.'
'>>> This is a great success.'
'>>> This is a great adventure.'
'>>> This is a great idea.'
'>>> This is a great feat.'


In [7]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
sample = imdb_dataset["train"].shuffle(seed=42).select(range(3))

for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...'
'>>> Label: 1'

'>>> Review: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stu

> ✏️ **Try it out!** Create a random sample of the `unsupervised` split and verify that the labels are neither `0` nor `1`. While you're at it, you could also check that the labels in the `train` and `test` splits are indeed `0` or `1` -- this is a useful sanity check that every NLP practitioner should perform at the start of a new project!

In [9]:
sample = imdb_dataset["test"].shuffle(seed=42).select(range(3))

for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it. I truly hope A Thousand Acres has had the same effect on some others out there.<br 

In [10]:
sample = imdb_dataset["unsupervised"].shuffle(seed=42).select(range(3))

for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: If you've seen the classic Roger Corman version starring Vincent Price it's hard to put it out of your head, but you probably should do because this one is totally different. Subtlety has been abandoned in favour of gross-out horror - nudity, gore and all-round unpleasantness. OK it's ridiculous, trashy, sensationalised and historically dubious (did any members of the Inquisition really wear horn-rimmed glasses?), but despite all this it is strangely compelling. I literally couldn't tear myself away from the screen until the end of the movie. If there's a bigger compliment you can pay to a film I don't know what it is.'
'>>> Label: -1'

'>>> Review: For me, this was the most moving film of the decade. Samira Makhmalbaf shows pure bravery and vision in the making. She has an intelligence and gift for speaking to the people, regardless of their nationality or beliefs. I am inspired and touched by her humanity and can only hope that she has touched many people the same way. 

In [11]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result


# Use batched=True to activate fast multithreading!
tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched=True, remove_columns=["text", "label"]
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

In [12]:
tokenizer.model_max_length

512

> ✏️ **Try it out!** Some Transformer models, like [BigBird](https://huggingface.co/google/bigbird-roberta-base) and [Longformer](https://huggingface.co/allenai/longformer-base-4096), have a much longer context length than BERT and other early Transformer models. Instantiate the tokenizer for one of these checkpoints and verify that the `model_max_length` agrees with what's quoted on its model card.

In [13]:
bigbird_tokenizer = AutoTokenizer.from_pretrained("google/bigbird-roberta-base")
longformer_tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
print(f">>> BigBird max length: {bigbird_tokenizer.model_max_length}")
print(f">>> Longformer max length: {longformer_tokenizer.model_max_length}")

>>> BigBird max length: 4096
>>> Longformer max length: 1000000000000000019884624838656


In [14]:
chunk_size = 128

In [15]:
# Slicing produces a list of lists for each feature
tokenized_samples = tokenized_datasets["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Review {idx} length: {len(sample)}'")

'>>> Review 0 length: 363'
'>>> Review 1 length: 304'
'>>> Review 2 length: 133'


In [16]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples["input_ids"])
print(f"'>>> Concatenated reviews length: {total_length}'")

'>>> Concatenated reviews length: 800'


In [17]:
chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")

'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 32'


In [18]:
def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    # Create a new labels column
    result["labels"] = result["input_ids"].copy()
    return result

In [19]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [20]:
tokenizer.decode(lm_datasets["train"][1]["input_ids"])

"as the vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men. < br / > < br / > what kills me about i am curious - yellow is that 40 years ago, this was considered pornographic. really, the sex and nudity scenes are few and far between, even then it ' s not shot like some cheaply made porno. while my countrymen mind find it shocking, in reality sex and nudity are a major staple in swedish cinema. even ingmar bergman,"

In [21]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [22]:
samples = [lm_datasets["train"][i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am [MASK] [MASK] yellow from my video store because [MASK] all the controversy [MASK] surrounded it when it was first released in 1967 [MASK] i also heard that [MASK] first it was seized [MASK] u. s. customs if it ever tried to [MASK] [MASK] [MASK], therefore being a [MASK] [MASK] films considered " controversial " i really had to see this for myself [MASK] < [MASK] [MASK] > < br / > the [MASK] is [MASK] around a young swedish drama student named pond who [MASK] [MASK] learn everything [MASK] can about life. in particular she wants to focus herurings to making some sort of documentary on what the average swede thought about [MASK] political issues [MASK]'

'>>> as the vietnam war and race issues in the united [MASK]. in between asking politicians and ordinary denizens of [MASK] about their opinions on politics, she has sex with her drama teacher, classmates, and married men. < br / > < [MASK] / > what kills me about i am curious - yellow [MASK] that 40 years [MAS

> ✏️ **Try it out!** Run the code snippet above several times to see the random masking happen in front of your very eyes! Also replace the `tokenizer.decode()` method with `tokenizer.convert_ids_to_tokens()` to see that sometimes a single token from a given word is masked, and not the others.

In [23]:
for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - yellow from my video store because of all [MASK] controversy that surrounded it when it was first released in 1967. [MASK] also heard [MASK] at [MASK] it was seized by u [MASK] s. [MASK] if it ever tried to enter this country, therefore being a fan of films considered " controversial " i really had to see this for myself. < [MASK] / > < br [MASK] > the plot is centered around a young swedish drama student [MASK] lena who wants to learn [MASK] she [MASK] about life. in particular she [MASK] to focus her attentions [MASK] making some sort of [MASK] on what [MASK] average swede thought [MASK] certain political [MASK] such'

'>>> as the vietnam war and race issues in the united states. in between asking fc and ordinary den 1856ns of [MASK] [MASK] their opinions on politics, she has sex with her drama teacher, classmates, [MASK] married men. [MASK] br / > < br / > what kills me about i am curious - yellow is that 40 years ago, this was considered pornogra

In [24]:
for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.convert_ids_to_tokens(chunk)}'")


'>>> ['[CLS]', 'i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', '[MASK]', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', '[MASK]', 'was', 'first', 'released', 'in', '1967', '.', 'i', 'also', 'heard', 'that', 'at', '[MASK]', 'it', '[MASK]', 'seized', 'by', 'u', '.', 's', '.', 'customs', 'if', 'it', 'ever', '[MASK]', 'to', 'enter', 'this', 'country', ',', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', '[MASK]', 'controversial', '"', 'i', '[MASK]', 'had', 'to', 'see', 'this', '[MASK]', 'myself', '.', '<', 'br', '/', '>', '<', 'br', '/', '##ek', '[MASK]', 'plot', 'is', 'centered', 'around', 'a', 'young', 'swedish', '[MASK]', 'student', 'named', 'lena', '[MASK]', 'wants', 'to', 'learn', 'everything', 'she', 'can', 'about', 'life', '.', 'in', 'particular', 'she', 'wants', 'to', 'focus', 'her', 'attention', '##s', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average', 'sw', '##ede', 'th

In [25]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probability = 0.2


def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # Create a map between words and corresponding token indices
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        # Randomly mask words
        mask = np.random.binomial(1, wwm_probability, (len(mapping),))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

In [26]:
samples = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] [MASK] rented i am curious - yellow from my video store because [MASK] all the controversy that surrounded it when it was [MASK] released [MASK] 1967. i also heard that at first it was seized [MASK] u. s. customs if it ever tried to enter this [MASK], [MASK] being a fan of films considered [MASK] controversial [MASK] i really had to [MASK] [MASK] for myself [MASK] [MASK] br / > [MASK] br [MASK] > [MASK] plot is centered [MASK] [MASK] young swedish drama student named lena who wants to learn everything she [MASK] about life. in particular she wants to focus her [MASK] [MASK] to making some sort of documentary [MASK] what [MASK] average swede thought about certain [MASK] [MASK] such'

'>>> as the [MASK] war and race issues in the united states. in between [MASK] [MASK] [MASK] ordinary denizens [MASK] stockholm [MASK] their opinions on politics, she has sex with her drama teacher [MASK] classmates, and married men. < br / > < br / > what [MASK] [MASK] [MASK] i am [MASK] [MASK]

> ✏️ **Try it out!** Run the code snippet above several times to see the random masking happen in front of your very eyes! Also replace the `tokenizer.decode()` method with `tokenizer.convert_ids_to_tokens()` to see that the tokens from a given word are always masked together.

In [27]:
samples = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)
for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - yellow [MASK] my video store because of all the controversy that surrounded it [MASK] it [MASK] first [MASK] in 1967. i also heard that [MASK] first it [MASK] seized by u. [MASK] [MASK] customs if it ever tried to enter [MASK] country, therefore being a fan of [MASK] considered " [MASK] " i really had [MASK] see this for myself. < [MASK] / > < br / > the plot is [MASK] around a young swedish drama student named lena who wants [MASK] learn everything she can about [MASK]. [MASK] particular she wants to focus her [MASK] [MASK] [MASK] making some sort of documentary on [MASK] the average swede [MASK] about certain political [MASK] such'

'>>> as the vietnam war and [MASK] issues in the united states. in between [MASK] politicians and ordinary [MASK] [MASK] [MASK] of stockholm [MASK] [MASK] opinions on politics, she has sex [MASK] her [MASK] teacher, classmates, and married men. < br / > < [MASK] / > what kills me about i am curious - yellow is that 40 y

In [28]:
samples = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)
for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am [MASK] - yellow from my [MASK] store because [MASK] all the controversy that surrounded it when [MASK] was first released in 1967 [MASK] i [MASK] [MASK] that at first [MASK] was seized by [MASK] [MASK] [MASK]. customs [MASK] it [MASK] tried to enter this country [MASK] [MASK] being a fan of films [MASK] [MASK] controversial [MASK] i really had [MASK] see this for myself. [MASK] br [MASK] > [MASK] br / > the [MASK] is centered around a young swedish drama student named lena who wants to [MASK] everything she [MASK] about life. in particular she wants [MASK] focus her attentions to making some sort of documentary on what the average [MASK] [MASK] [MASK] about certain [MASK] issues such'

'>>> [MASK] [MASK] vietnam war and race [MASK] in the united states. [MASK] between asking politicians and ordinary denizens of stockholm about their opinions on politics, she [MASK] sex with her drama teacher, [MASK] [MASK] and married men [MASK] < [MASK] / [MASK] [MASK] br [MA

In [29]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

In [30]:
from huggingface_hub import notebook_login

notebook_login()

In [31]:
from transformers import TrainingArguments

batch_size = 64
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
)

In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [33]:
import math

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
No log,3.088143,0


>>> Perplexity: 21.94


In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.681751,2.512154
2,2.591912,2.449711
3,2.528859,2.482906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.6002666773057035, metrics={'train_runtime': 113.1841, 'train_samples_per_second': 265.055, 'train_steps_per_second': 4.161, 'total_flos': 994208670720000.0, 'train_loss': 2.6002666773057035, 'epoch': 3.0})

In [35]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
2.528859,2.490135,3


>>> Perplexity: 12.06


In [36]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hwting/distilbert-base-uncased-finetuned-imdb/commit/fa9e860311dbaaf57aeea8cb464309fd3cae6792', commit_message='End of training', commit_description='', oid='fa9e860311dbaaf57aeea8cb464309fd3cae6792', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hwting/distilbert-base-uncased-finetuned-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='hwting/distilbert-base-uncased-finetuned-imdb'), pr_revision=None, pr_num=None)

> ✏️ **Your turn!** Run the training above after changing the data collator to the whole word masking collator. Do you get better results?

In [37]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb-whole-word-masking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=whole_word_masking_data_collator,
    processing_class=tokenizer,
)

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
No log,4.134855,0


>>> Perplexity: 62.48


In [38]:
trainer.train()
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Epoch,Training Loss,Validation Loss
1,3.560645,3.319996
2,3.406486,3.287030
3,3.371273,3.279518


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
3.371273,3.306160,3


>>> Perplexity: 27.28


In [39]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hwting/distilbert-base-uncased-finetuned-imdb-whole-word-masking/commit/4e86c94dd9493251041b9a25e5630c0d9cf4a9ff', commit_message='End of training', commit_description='', oid='4e86c94dd9493251041b9a25e5630c0d9cf4a9ff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hwting/distilbert-base-uncased-finetuned-imdb-whole-word-masking', endpoint='https://huggingface.co', repo_type='model', repo_id='hwting/distilbert-base-uncased-finetuned-imdb-whole-word-masking'), pr_revision=None, pr_num=None)

In [40]:
def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

In [41]:
downsampled_dataset = downsampled_dataset.remove_columns(["word_ids"])
eval_dataset = downsampled_dataset["test"].map(
    insert_random_mask,
    batched=True,
    remove_columns=downsampled_dataset["test"].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [42]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

batch_size = 64
train_dataloader = DataLoader(
    downsampled_dataset["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)

In [43]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [44]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [45]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [46]:
from huggingface_hub import HfApi, get_full_repo_name

hf_api = HfApi()
model_name = "distilbert-base-uncased-finetuned-imdb-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'hwting/distilbert-base-uncased-finetuned-imdb-accelerate'

In [47]:
from huggingface_hub import create_repo

output_dir = model_name
create_repo(repo_name, exist_ok=True)

RepoUrl('https://huggingface.co/hwting/distilbert-base-uncased-finetuned-imdb-accelerate', endpoint='https://huggingface.co', repo_type='model', repo_id='hwting/distilbert-base-uncased-finetuned-imdb-accelerate')

In [48]:
from tqdm.auto import tqdm
import torch
import math

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataset)]
    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        hf_api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_name,
            commit_message=f"Training in progress epoch {epoch}",
            run_as_future=True,
        )

  0%|          | 0/471 [00:00<?, ?it/s]

>>> Epoch 0: Perplexity: 10.879524577627729


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

>>> Epoch 2: Perplexity: 10.399711235372179


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [49]:
from transformers import pipeline

mask_filler = pipeline(
    "fill-mask", model="huggingface-course/distilbert-base-uncased-finetuned-imdb"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [50]:
preds = mask_filler(text)

for pred in preds:
    print(f">>> {pred['sequence']}")

>>> this is a great film.
>>> this is a great movie.
>>> this is a great idea.
>>> this is a great deal.
>>> this is a great adventure.


> ✏️ **Try it out!** To quantify the benefits of domain adaptation, fine-tune a classifier on the IMDb labels for both the pretrained and fine-tuned DistilBERT checkpoints. If you need a refresher on text classification, check out [Chapter 3](/course/chapter3).

In [51]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
del imdb_dataset["unsupervised"]
tokenized_datasets = imdb_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [52]:
import evaluate

torch.cuda.empty_cache()
pretrained_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="pretrained-distilbert-imdb-classification",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        **accuracy.compute(predictions=preds, references=labels),
        **f1.compute(predictions=preds, references=labels)
    }

trainer = Trainer(
    pretrained_model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [53]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.241864,0.198319,0.923000,0.921310
2,0.160891,0.189994,0.929160,0.929400
3,0.109673,0.203452,0.928440,0.928875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1173, training_loss=0.18088715724831245, metrics={'train_runtime': 1163.6255, 'train_samples_per_second': 64.454, 'train_steps_per_second': 1.008, 'total_flos': 9935054899200000.0, 'train_loss': 0.18088715724831245, 'epoch': 3.0})

In [54]:
eval_results = trainer.evaluate()
print(f">>> Accuracy for pretrained model: {eval_results['eval_accuracy']:.2f}")
print(f">>> F1 for pretrained model: {eval_results['eval_f1']:.2f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.109673,0.203452,3,0.928440,0.928875


>>> Accuracy for pretrained model: 0.93
>>> F1 for pretrained model: 0.93


In [55]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hwting/pretrained-distilbert-imdb-classification/commit/54d00000f30e6abdfa95e979c7c2bc65a83299a9', commit_message='End of training', commit_description='', oid='54d00000f30e6abdfa95e979c7c2bc65a83299a9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hwting/pretrained-distilbert-imdb-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='hwting/pretrained-distilbert-imdb-classification'), pr_revision=None, pr_num=None)

In [56]:
torch.cuda.empty_cache()
finetuned_checkpoint = "hwting/distilbert-base-uncased-finetuned-imdb"
finetuned_model = AutoModelForSequenceClassification.from_pretrained(finetuned_checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="fintuned-distilbert-imdb-classification",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps
)

trainer = Trainer(
    finetuned_model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: hwting/distilbert-base-uncased-finetuned-imdb
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [57]:
trainer.train()
eval_results = trainer.evaluate()
print(f">>> Accuracy for domain adaptation model: {eval_results['eval_accuracy']:.2f}")
print(f">>> F1 for domain adaptation model: {eval_results['eval_f1']:.2f}")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.243290,0.200995,0.921840,0.920088
2,0.164449,0.186659,0.931240,0.931072
3,0.111677,0.201311,0.929800,0.930216


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.111677,0.201311,3,0.929800,0.930216


>>> Accuracy for domain adaptation model: 0.93
>>> F1 for domain adaptation model: 0.93


In [58]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hwting/fintuned-distilbert-imdb-classification/commit/d64061cc578318a68d40d87b13189e842de8955f', commit_message='End of training', commit_description='', oid='d64061cc578318a68d40d87b13189e842de8955f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hwting/fintuned-distilbert-imdb-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='hwting/fintuned-distilbert-imdb-classification'), pr_revision=None, pr_num=None)